## Cell 1

In [ ]:
%pip install -q chromadb sentence-transformers numpy scikit-learn

## Cell 2

In [ ]:
import chromadb
import numpy as np
import sklearn

from sentence_transformers import SentenceTransformer

print("ChromaDB:", chromadb.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

print("All project libraries imported successfully!")

## Cell 3

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cpu"
)

embedding = model.encode(
    "Artificial intelligence is fascinating."
)

print("Embedding dimensions:", len(embedding))

## Cell 4

In [ ]:
import chromadb
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

## Cell 5

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cpu"
)

print("Embedding model loaded!")

## Cell 6

In [ ]:
text = "Artificial intelligence is transforming technology."

embedding = model.encode(text)

print("Embedding dimensions:", len(embedding))
print("First 10 values:", embedding[:10])

## Cell 7

In [ ]:
client = chromadb.PersistentClient(
    path="/content/chroma_database"
)

print("ChromaDB database created!")

## Cell 8

In [ ]:
collection = client.get_or_create_collection(
    name="library_books",
    metadata={
        "hnsw:space": "cosine"
    }
)

print("Collection created!")

## Cell 9

In [ ]:
books = [
    "Animal Farm is a political allegory about "
    "revolution, authoritarianism and propaganda.",

    "1984 explores totalitarianism, surveillance, "
    "censorship and the manipulation of truth.",

    "The Hobbit is a fantasy adventure about "
    "a hobbit who joins a quest involving dragons.",

    "Jane Eyre explores love, independence, "
    "morality and social class.",

    "Brave New World examines technological control, "
    "consumerism and social conditioning."
]

book_ids = ["book1", "book2", "book3", "book4", "book5"]

book_metadata = [
    {"title": "Animal Farm", "author": "George Orwell", "genre": "Political fiction"},
    {"title": "1984", "author": "George Orwell", "genre": "Dystopian fiction"},
    {"title": "The Hobbit", "author": "J.R.R. Tolkien", "genre": "Fantasy"},
    {"title": "Jane Eyre", "author": "Charlotte Bronte", "genre": "Classic fiction"},
    {"title": "Brave New World", "author": "Aldous Huxley", "genre": "Dystopian fiction"}
]

book_embeddings = model.encode(
    books,
    normalize_embeddings=True
).tolist()

collection.upsert(
    ids=book_ids,
    documents=books,
    embeddings=book_embeddings,
    metadatas=book_metadata
)

print("Books inserted successfully!")

## Cell 10

In [ ]:
results = collection.get(
    include=["documents", "metadatas", "embeddings"]
)

for i in range(len(results["ids"])):

    print("ID:", results["ids"][i])
    print("Document:", results["documents"][i])
    print("Metadata:", results["metadatas"][i])
    print("-" * 40)

## Cell 11

In [ ]:
result = collection.get(
    ids=["book1"],
    include=["documents", "metadatas"]
)

print(result)

## Cell 12

In [ ]:
new_description = (
    "Animal Farm is a satirical novella about "
    "the corruption of political revolutions, "
    "dictatorship and the abuse of power."
)

new_embedding = model.encode(
    new_description,
    normalize_embeddings=True
).tolist()

collection.update(
    ids=["book1"],
    documents=[new_description],
    embeddings=[new_embedding],
    metadatas=[{
        "title": "Animal Farm",
        "author": "George Orwell",
        "genre": "Political satire"
    }]
)

print("Book updated successfully!")

## Cell 13

In [ ]:
collections = client.list_collections()

print("Available collections:")

for item in collections:
    print(item)

## Cell 14

In [ ]:
collection = client.get_collection(
    name="library_books"
)

## Cell 15

In [ ]:
sentences = [
    "Artificial intelligence and machine learning.",
    "Deep learning uses neural networks.",
    "The history of medieval European architecture.",
    "Neural networks are used in artificial intelligence.",
    "The principles of classical economics."
]

sentence_embeddings = model.encode(
    sentences,
    normalize_embeddings=True
)

query = "How do artificial intelligence systems learn?"

query_embedding = model.encode(
    [query],
    normalize_embeddings=True
)

## Cell 16

In [ ]:
scores = cosine_similarity(
    query_embedding,
    sentence_embeddings
)[0]

ranked_indices = np.argsort(scores)[::-1]

for index in ranked_indices:

    print("Sentence:", sentences[index])
    print("Similarity:", round(scores[index], 4))
    print("-" * 40)

## Cell 17

In [ ]:
search_collection = client.get_or_create_collection(
    name="semantic_search_demo",
    metadata={"hnsw:space": "cosine"}
)

search_collection.upsert(
    ids=[f"sentence_{i}" for i in range(len(sentences))],
    documents=sentences,
    embeddings=sentence_embeddings.tolist()
)

print("Search collection ready!")

## Cell 18

In [ ]:
query = "How do artificial intelligence systems learn?"

query_vector = model.encode(
    query,
    normalize_embeddings=True
).tolist()

results = search_collection.query(
    query_embeddings=[query_vector],
    n_results=3,
    include=["documents", "distances"]
)

for i in range(len(results["ids"][0])):

    distance = results["distances"][0][i]

    print("Document:", results["documents"][0][i])
    print("Cosine distance:", round(distance, 4))
    print("Cosine similarity:", round(1 - distance, 4))
    print("-" * 40)

## Cell 19

In [ ]:
results = collection.query(
    query_embeddings=[
        model.encode(
            "A novel about political control and propaganda.",
            normalize_embeddings=True
        ).tolist()
    ],
    n_results=3,
    where={"genre": "Dystopian fiction"},
    include=["documents", "metadatas", "distances"]
)

## Cell 20

In [ ]:
recommendation_books = [

    {
        "id": "r1",
        "title": "1984",
        "description": "A dystopian novel about surveillance, "
                       "totalitarianism and government control."
    },

    {
        "id": "r2",
        "title": "Animal Farm",
        "description": "A political satire about revolution, "
                       "authoritarianism and propaganda."
    },

    {
        "id": "r3",
        "title": "Brave New World",
        "description": "A dystopian society controlled through "
                       "technology, conditioning and consumerism."
    },

    {
        "id": "r4",
        "title": "Fahrenheit 451",
        "description": "A dystopian novel about censorship, "
                       "book burning and intellectual freedom."
    },

    {
        "id": "r5",
        "title": "The Hobbit",
        "description": "A fantasy adventure about dragons, "
                       "friendship and an unexpected journey."
    },

    {
        "id": "r6",
        "title": "Jane Eyre",
        "description": "A classic novel exploring independence, "
                       "love, morality and social class."
    },

    {
        "id": "r7",
        "title": "The Handmaid's Tale",
        "description": "A dystopian novel about authoritarian "
                       "rule, surveillance and restrictions on freedom."
    }
]

print("Books loaded:", len(recommendation_books))

## Cell 21

In [ ]:
book_texts = [
    book["description"]
    for book in recommendation_books
]

book_vectors = model.encode(
    book_texts,
    normalize_embeddings=True
)

print("Embeddings generated!")
print("Number of books:", len(book_vectors))
print("Vector dimensions:", book_vectors.shape[1])

## Cell 22

In [ ]:
recommendation_collection = client.get_or_create_collection(
    name="book_recommendations",
    metadata={"hnsw:space": "cosine"}
)

recommendation_collection.upsert(
    ids=[book["id"] for book in recommendation_books],

    documents=book_texts,

    embeddings=book_vectors.tolist(),

    metadatas=[
        {
            "title": book["title"]
        }
        for book in recommendation_books
    ]
)

print("Recommendation database created!")

## Cell 23

In [ ]:
def recommend_books(liked_ids, top_k=3):

    # Find the vectors of books the user likes
    liked_indices = [
        i for i, book in enumerate(recommendation_books)
        if book["id"] in liked_ids
    ]

    if not liked_indices:
        print("No recognised liked books.")
        return

    # Create a user preference vector
    user_vector = np.mean(
        book_vectors[liked_indices],
        axis=0
    )

    # Search the vector database
    results = recommendation_collection.query(
        query_embeddings=[user_vector.tolist()],
        n_results=len(recommendation_books),
        include=["metadatas", "distances"]
    )

    print("RECOMMENDED BOOKS\n")

    count = 0

    for i, book_id in enumerate(results["ids"][0]):

        # Exclude books already liked
        if book_id in liked_ids:
            continue

        title = results["metadatas"][0][i]["title"]
        distance = results["distances"][0][i]

        print("Book:", title)
        print("Similarity:", round(1 - distance, 4))
        print("-" * 40)

        count += 1

        if count >= top_k:
            break

## Cell 24

In [ ]:
recommend_books(
    liked_ids=["r1", "r2"],
    top_k=3
)